# Демонстрация распространения орбиты

Демонстрация распространения орбиты с проверкой по реальным точным орбитальным положениям GNSS SP3.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Gavr101/space_modeling.git'


def run(cmd):
    print('>>', ' '.join(cmd))
    subprocess.check_call(cmd)


def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'dynamics').is_dir() and (candidate / 'visualization').is_dir():
            return candidate
    raise RuntimeError(f'Cannot find project root from {start}')


if IN_COLAB:
    PROJECT_ROOT = Path('/content/space_modeling')
    if not PROJECT_ROOT.exists():
        run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
    os.chdir(PROJECT_ROOT)

    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
    run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
    run([sys.executable, '-m', 'pip', 'install', '-e', '.'])
    # Опционально: NRLMSISE-00 улучшает моделирование плотности для сопротивления, если доступна.
    try:
        run([sys.executable, '-m', 'pip', 'install', 'nrlmsise00'])
    except Exception as exc:
        print('Optional dependency nrlmsise00 was not installed:', exc)
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Working dir:', Path.cwd())


Working dir: C:\Users\Gavriil\VS_projects\space_modeling


## Что здесь моделируется

В ноутбуке используются:
- `dynamics.propagator.PropagationConfig` для параметров распространения;
- `dynamics.propagator.propagate_orbit` для численного распространения орбиты;
- `dynamics.sp3` и `dynamics.eof` для чтения эталонных состояний точных орбит;
- `visualization.orbit_3d.build_orbit_figure` для 3D-графиков орбиты;
- `visualization.map_2d.build_groundtrack_figure` для 2D-трасс подспутниковой точки.

Набор проверки по умолчанию использует продукты точных орбит, содержащие исходные записи скоростей. Продукты SP3 со строками `V` и продукты Sentinel EOF/OSV преобразуются из земной ITRS/ECEF в GCRS с помощью Astropy. Продукты GNSS SP3 только с положениями хранятся отдельно как опциональная грубая проверка орбит, но не используются в основном сценарии отладки скоростей.

### Начальные условия и параметры
Распространение задаётся следующими параметрами:
- `initial_state = [x, y, z, vx, vy, vz]`;
- `epoch_seconds`, `duration_seconds`, `step_seconds`, `integrator`;
- свойства космического аппарата: `mass`, `cd`, `cr`, `reference_area`.

## Эталонные данные точных орбит

В примерах ниже используются публичные продукты точных орбит с исходными векторами состояния. Файлы ESA Swarm POD SP3 содержат записи скоростей Swarm A/B/C (строки `#dV` и `V`). CASSIOPE/Swarm-E предоставляет суточный файл SP3 с положениями и скоростями с частотой 1 Гц. Файлы Sentinel-1 POEORB предоставляют EOF/XML Orbit State Vectors с полями `X, Y, Z, VX, VY, VZ`. GNSS CODE MGEX остаётся доступным как опциональный источник только положений, но не входит в эталонный набор по умолчанию, потому что скорости для него пришлось бы оценивать конечными разностями.

In [2]:
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'dynamics').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dynamics import make_recommended_force_config
from dynamics.eof import eof_state_samples, read_sentinel_eof
from dynamics.propagator import PropagationConfig, SpacecraftProperties, propagate_orbit
from dynamics.sp3 import download_sp3, read_sp3, sp3_state_samples
from visualization.orbit_3d import build_orbit_figure
from visualization.map_2d import build_groundtrack_figure


In [3]:
VELOCITY_REFERENCE_SOURCES = {
    'Sentinel-1A': {
        'format': 'sentinel_eof',
        'url': 'https://s1-orbits.s3.us-west-2.amazonaws.com/AUX_POEORB/S1A_OPER_AUX_POEORB_OPOD_20230821T080724_V20230731T225942_20230802T005942.EOF',
        'path': PROJECT_ROOT / 'data' / 'eof' / 'S1A_OPER_AUX_POEORB_OPOD_20230821T080724_V20230731T225942_20230802T005942.EOF',
        'requires_velocity': True,
        'velocity_source': 'sentinel_eof_osv',
        'spacecraft': SpacecraftProperties(mass=2185.0, cd=2.2, cr=1.3, reference_area=10.3),
    },
    'Sentinel-1B': {
        'format': 'sentinel_eof',
        'url': 'https://s1-orbits.s3.us-west-2.amazonaws.com/AUX_POEORB/S1B_OPER_AUX_POEORB_OPOD_20210203T112353_V20210113T225942_20210115T005942.EOF',
        'path': PROJECT_ROOT / 'data' / 'eof' / 'S1B_OPER_AUX_POEORB_OPOD_20210203T112353_V20210113T225942_20210115T005942.EOF',
        'requires_velocity': True,
        'velocity_source': 'sentinel_eof_osv',
        'spacecraft': SpacecraftProperties(mass=2185.0, cd=2.2, cr=1.3, reference_area=10.3),
    },
    'CASSIOPE / Swarm-E': {
        'format': 'sp3',
        'satellite_id': 'L63',
        'url': 'https://epop-data.phys.ucalgary.ca/2017/02/01/CAS_Orbit_GEO_20170201T000000_20170201T235959_1.1.0.sp3.zip',
        'path': PROJECT_ROOT / 'data' / 'sp3' / 'CAS_Orbit_GEO_20170201T000000_20170201T235959_1.1.0.sp3.zip',
        'requires_velocity': True,
        'velocity_source': 'sp3_velocity_records',
        'spacecraft': SpacecraftProperties(mass=500.0, cd=2.2, cr=1.3, reference_area=2.2),
    },
    'Swarm B': {
        'format': 'sp3',
        'satellite_id': 'L48',
        'url': 'https://swarm-diss.eo.esa.int/?do=download&file=swarm%2FLevel2daily%2FLatest_baselines%2FPOD%2FRD%2FSat_B%2FSW_OPER_SP3BCOM_2__20240106T235942_20240107T235942_0201.ZIP',
        'path': PROJECT_ROOT / 'data' / 'sp3' / 'SW_OPER_SP3BCOM_2__20240106T235942_20240107T235942_0201.ZIP',
        'requires_velocity': True,
        'velocity_source': 'sp3_velocity_records',
        'spacecraft': SpacecraftProperties(mass=438.0, cd=2.2, cr=1.3, reference_area=1.1),
    },
    'Swarm C': {
        'format': 'sp3',
        'satellite_id': 'L49',
        'url': 'https://swarm-diss.eo.esa.int/?do=download&file=swarm%2FLevel2daily%2FLatest_baselines%2FPOD%2FRD%2FSat_C%2FSW_OPER_SP3CCOM_2__20240106T235942_20240107T235942_0201.ZIP',
        'path': PROJECT_ROOT / 'data' / 'sp3' / 'SW_OPER_SP3CCOM_2__20240106T235942_20240107T235942_0201.ZIP',
        'requires_velocity': True,
        'velocity_source': 'sp3_velocity_records',
        'spacecraft': SpacecraftProperties(mass=438.0, cd=2.2, cr=1.3, reference_area=1.1),
    },
    'Swarm A': {
        'format': 'sp3',
        'satellite_id': 'L47',
        'url': 'https://swarm-diss.eo.esa.int/?do=download&file=swarm%2FLevel2daily%2FLatest_baselines%2FPOD%2FRD%2FSat_A%2FSW_OPER_SP3ACOM_2__20240106T235942_20240107T235942_0201.ZIP',
        'path': PROJECT_ROOT / 'data' / 'sp3' / 'SW_OPER_SP3ACOM_2__20240106T235942_20240107T235942_0201.ZIP',
        'requires_velocity': True,
        'velocity_source': 'sp3_velocity_records',
        'spacecraft': SpacecraftProperties(mass=438.0, cd=2.2, cr=1.3, reference_area=1.1),
    },
}

POSITION_ONLY_GNSS_SOURCES = {
    'QZSS J04': {'format': 'sp3', 'satellite_id': 'J04', 'url': 'ftp://igs.ign.fr/pub/igs/products/2296/COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz', 'path': PROJECT_ROOT / 'data' / 'sp3' / 'COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz'},
    'BeiDou C09': {'format': 'sp3', 'satellite_id': 'C09', 'url': 'ftp://igs.ign.fr/pub/igs/products/2296/COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz', 'path': PROJECT_ROOT / 'data' / 'sp3' / 'COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz'},
    'QZSS J03': {'format': 'sp3', 'satellite_id': 'J03', 'url': 'ftp://igs.ign.fr/pub/igs/products/2296/COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz', 'path': PROJECT_ROOT / 'data' / 'sp3' / 'COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz'},
    'Galileo E14': {'format': 'sp3', 'satellite_id': 'E14', 'url': 'ftp://igs.ign.fr/pub/igs/products/2296/COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz', 'path': PROJECT_ROOT / 'data' / 'sp3' / 'COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz'},
    'BeiDou C11': {'format': 'sp3', 'satellite_id': 'C11', 'url': 'ftp://igs.ign.fr/pub/igs/products/2296/COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz', 'path': PROJECT_ROOT / 'data' / 'sp3' / 'COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz'},
    'GPS G21': {'format': 'sp3', 'satellite_id': 'G21', 'url': 'ftp://igs.ign.fr/pub/igs/products/2296/COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz', 'path': PROJECT_ROOT / 'data' / 'sp3' / 'COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz'},
    'GLONASS R20': {'format': 'sp3', 'satellite_id': 'R20', 'url': 'ftp://igs.ign.fr/pub/igs/products/2296/COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz', 'path': PROJECT_ROOT / 'data' / 'sp3' / 'COD0MGXFIN_20240070000_01D_05M_ORB.SP3.gz'},
}

EARTH_RADIUS_M = 6_378_137.0
DURATION_HOURS = 12
STEP_SECONDS = 60.0


In [4]:
def download_reference_file(source):
    return download_sp3(source['url'], source['path'])


def reference_has_native_velocity(source, path):
    if source['format'] == 'sp3':
        orbit = read_sp3(path)
        return source['satellite_id'] in getattr(orbit, 'velocities_m_s', {})
    if source['format'] == 'sentinel_eof':
        orbit = read_sentinel_eof(path)
        return orbit.velocities_m_s.shape[0] >= 2
    raise ValueError(f'Unsupported reference format: {source["format"]}')


def load_reference_orbit(source, duration_hours=DURATION_HOURS, step_seconds=STEP_SECONDS):
    path = download_reference_file(source)
    has_velocity = reference_has_native_velocity(source, path)
    if source.get('requires_velocity', False) and not has_velocity:
        raise ValueError(f'{path.name} does not contain native velocity records required for this source.')

    if source['format'] == 'sp3':
        epochs, states = sp3_state_samples(
            path,
            source['satellite_id'],
            duration_hours=duration_hours,
            step_seconds=step_seconds,
        )
    elif source['format'] == 'sentinel_eof':
        epochs, states = eof_state_samples(
            path,
            duration_hours=duration_hours,
            step_seconds=step_seconds,
        )
    else:
        raise ValueError(f'Unsupported reference format: {source["format"]}')
    return path, epochs, states, has_velocity


def median_altitude_km(states):
    return float(np.median(np.linalg.norm(states[:, :3], axis=1) - EARTH_RADIUS_M) / 1000.0)


def run_model_from_initial_reference_state(
    initial_state,
    start_epoch,
    duration_hours=DURATION_HOURS,
    step_seconds=STEP_SECONDS,
    spacecraft_prop=SpacecraftProperties(500.0, 2.2, 1.3, 2.2),
):
    cfg = PropagationConfig(
        initial_state=initial_state,
        epoch_seconds=float(start_epoch.unix),
        duration_seconds=duration_hours * 3600.0,
        step_seconds=step_seconds,
        integrator='dop853',
        spacecraft=spacecraft_prop,
    )
    cfg.environment.force_models = make_recommended_force_config()
    space_weather_file = PROJECT_ROOT / 'data' / 'cache' / 'SW-All.csv'
    if space_weather_file.exists():
        cfg.environment.space_weather_file = space_weather_file
    _, model_states = propagate_orbit(cfg)
    return model_states


In [5]:
reference_runs = []

for sat_name, source in VELOCITY_REFERENCE_SOURCES.items():
    ref_path, ref_epochs, ref_states, has_velocity = load_reference_orbit(source)
    actual_duration_hours = (ref_states.shape[0] - 1) * STEP_SECONDS / 3600.0
    model_states = run_model_from_initial_reference_state(
        ref_states[0],
        ref_epochs[0],
        duration_hours=actual_duration_hours,
        step_seconds=STEP_SECONDS,
        spacecraft_prop=source['spacecraft'],
    )

    pos_err = np.linalg.norm(model_states[:, :3] - ref_states[:, :3], axis=1)
    vel_err = np.linalg.norm(model_states[:, 3:] - ref_states[:, 3:], axis=1)
    reference_runs.append(
        {
            'name': sat_name,
            'source': source,
            'path': ref_path,
            'epochs': ref_epochs,
            'reference_states': ref_states,
            'model_states': model_states,
            'elapsed_seconds': np.arange(ref_states.shape[0], dtype=float) * STEP_SECONDS,
            'has_velocity': has_velocity,
            'median_altitude_km': median_altitude_km(ref_states),
            'max_position_error_km': float(pos_err.max() / 1000.0),
            'max_velocity_error_m_s': float(vel_err.max()),
        }
    )

reference_runs.sort(key=lambda item: item['median_altitude_km'], reverse=True)

tracks_eci = []
tracks_elapsed_seconds = []
names_3d = []

for run in reference_runs:
    tracks_eci.append(run['reference_states'][:, :3])
    tracks_elapsed_seconds.append(run['elapsed_seconds'])
    names_3d.append(f'{run["name"]} | reference')
    tracks_eci.append(run['model_states'][:, :3])
    tracks_elapsed_seconds.append(run['elapsed_seconds'])
    names_3d.append(f'{run["name"]} | model')

    print(f'{run["name"]}: source = {run["path"].name}, format = {run["source"]["format"]}')
    print(f'{run["name"]}: median altitude = {run["median_altitude_km"]:.1f} km, native velocity = {run["has_velocity"]}')
    print(f'{run["name"]}: max |dr| = {run["max_position_error_km"]:.1f} km, max |dv| = {run["max_velocity_error_m_s"]:.2f} m/s\n')


CASSIOPE / Swarm-E: source = CAS_Orbit_GEO_20170201T000000_20170201T235959_1.1.0.sp3.zip, format = sp3
CASSIOPE / Swarm-E: median altitude = 869.2 km, native velocity = True
CASSIOPE / Swarm-E: max |dr| = 1.7 km, max |dv| = 1.57 m/s

Sentinel-1B: source = S1B_OPER_AUX_POEORB_OPOD_20210203T112353_V20210113T225942_20210115T005942.EOF, format = sentinel_eof
Sentinel-1B: median altitude = 699.3 km, native velocity = True
Sentinel-1B: max |dr| = 2.3 km, max |dv| = 2.33 m/s

Sentinel-1A: source = S1A_OPER_AUX_POEORB_OPOD_20230821T080724_V20230731T225942_20230802T005942.EOF, format = sentinel_eof
Sentinel-1A: median altitude = 698.4 km, native velocity = True
Sentinel-1A: max |dr| = 2.9 km, max |dv| = 2.93 m/s

Swarm B: source = SW_OPER_SP3BCOM_2__20240106T235942_20240107T235942_0201.ZIP, format = sp3
Swarm B: median altitude = 513.4 km, native velocity = True
Swarm B: max |dr| = 2.4 km, max |dv| = 2.56 m/s

Swarm C: source = SW_OPER_SP3CCOM_2__20240106T235942_20240107T235942_0201.ZIP, format

In [14]:
fig3d = build_orbit_figure(states=tracks_eci, names=names_3d, show_earth=True)
fig3d.show()


In [ ]:
# fig3d.write_html(str(PROJECT_ROOT / 'history' / '3_changed_to_sp3' / '3d_orbit.html'))

## 2D трасса подспутниковой точки

Карта ниже использует существующий приближённый поворот ECI->ECEF вокруг оси Z Земли. Проверка по SP3 выше перед сравнением траекторий использует более корректное преобразование Astropy ITRS->GCRS.

In [15]:
fig2d = build_groundtrack_figure(
    trajectories_eci=tracks_eci,
    names=names_3d,
    elapsed_seconds=tracks_elapsed_seconds,
)
fig2d.show()


In [19]:
fig2d.write_html(str(PROJECT_ROOT / 'history' / '3_changed_to_sp3' / '2d_ground.html'))